This notebook demonstrates the training of a Bidirectional Encoder Representations from Transformers model to solve an issue in conversational AI agents used in chatbots: **End-of-Utterance Detection**.

**The Problem:** AI agents often wait for long silences to process user audio/text, increasing latency. Alternatively, they process fragmented messages, consuming unnecessary tokens in LLM calls.
**The Solution:** Train a lightweight binary classification model to act as a *gatekeeper*, analyzing text in real-time and returning `1` (complete sentence) or `0` (incomplete sentence).

In [ ]:
# Install required dependencies
!pip install -q transformers datasets evaluate scikit-learn pandas numpy

In [ ]:
import random
import re
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

# Replace with your write-access token
login("YOUR_TOKEN_HERE")

I use the `brazilian-customer-service-conversations` dataset. Since the goal is to identify incomplete sentences, i developed a *Data Augmentation* function (`extract_and_break_smart`) that intentionally cuts real sentences to generate negative examples (Label 0), while also injecting common short chat phrases.

In [ ]:
raw_dataset = load_dataset("RichardSakaguchiMS/brazilian-customer-service-conversations")

def extract_and_break_smart(batch):
    new_texts, new_labels = [], []

    for conversation in batch['messages']:
        for msg in conversation:
            if msg['role'] == 'customer':
                content = msg['content'].strip()
                words = content.split()
                # Example of minimum length threshold based on dataset characteristics
                if len(words) > 5:
                    # 1. LABEL 0
                    max_cut = int(len(words) * 0.7)
                    for _ in range(3):
                        cut_point = random.randint(2, max_cut)
                        incomplete = " ".join(words[:cut_point])
                        incomplete = re.sub(r'[.,?!]$', '', incomplete)
                        new_texts.append(incomplete)
                        new_labels.append(0)

                    # 2. LABEL 1
                    new_texts.append(content)
                    new_labels.append(1)

    return {"text": new_texts, "label": new_labels}

# Apply processing and remove duplicates
processed_data = raw_dataset['train'].map(
    extract_and_break_smart,
    batched=True,
    remove_columns=raw_dataset['train'].column_names
)
df = pd.DataFrame(processed_data).drop_duplicates(subset=['text'])

# Custom phrases specifically for the target chatbot context
extra_data = [
  {"text": "quero comprar minério", "label": 1},
  {"text": "eu queria fazer um", "label": 0},
  # ... [INSERT THE REST OF YOUR LIST HERE] ...
]
df_extra = pd.DataFrame(extra_data)

# Splitting and Oversampling
train_nvidia, test_nvidia = train_test_split(df, test_size=0.3, random_state=42)
train_ia, test_ia = train_test_split(df_extra, test_size=0.3, random_state=42)

# Multiplying by 20 to emphasize short chat interactions during training
train_ia_amplified = pd.concat([train_ia] * 20, ignore_index=True)

df_train_final = pd.concat([train_nvidia, train_ia_amplified]).sample(frac=1).reset_index(drop=True)
df_test_final = pd.concat([test_nvidia, test_ia]).sample(frac=1).reset_index(drop=True)

hf_train = Dataset.from_pandas(df_train_final)
hf_test = Dataset.from_pandas(df_test_final)

I use the `bert-base-portuguese-cased` tokenizer from Neuralmind, applying standard padding and truncation to ensure inference efficiency. A `max_length` of 64 is sufficient for typical chat messages.

In [ ]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tok_train = hf_train.map(tokenize_func, batched=True)
tok_test = hf_test.map(tokenize_func, batched=True)

For this use case, **Precision** is the most critical metric. A false positive, in the use case of a chatbot, will cause the issue that the model was designed to solving. The **F1-Score** is used as the primary metric for Early Stopping to ensure a balance with Recall.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

Implementation of a robust training loop with automatic overfitting control via `EarlyStoppingCallback`. The model is evaluated at the end of each epoch, and the best checkpoint is preserved based on the F1-Score.

In [ ]:
target_f1 = 0.85
current_f1 = 0.0
attempt = 1
current_lr = 5e-5
current_epochs = 3

while current_f1 < target_f1 and attempt <= 3:
    print(f"\n>>> ATTEMPT {attempt} | Target F1-Score: {target_f1} | Learning Rate: {current_lr}")
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    args = TrainingArguments(
        output_dir=f"./training_results/attempt_{attempt}",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=current_lr,
        num_train_epochs=current_epochs,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="tensorboard"
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_test,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    current_f1 = eval_results['eval_f1']
    
    print(f"Results (Attempt {attempt}):")
    print(f"F1-Score:  {current_f1:.4f} | Precision: {eval_results['eval_precision']:.4f} | Recall: {eval_results['eval_recall']:.4f}")
    
    if current_f1 >= target_f1:
        print(f"train finished")
        break
    else:
        print(f"train not finished")
        current_lr /= 2
        current_epochs += 2
        attempt += 1